<a href="https://colab.research.google.com/github/Samu3lNM/Projects/blob/main/Finance_Automation_Monthly_Revenue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Finance Automation using CRISP-DM

## 1. Business Understanding
**Objective:** Automate the monthly revenue reporting for the Finance department to reduce manual effort and provide timely financial insights.
**Goal:** Generate a CSV report containing monthly revenue aggregates from raw transaction data.

# Finance Automation: Monthly Revenue Reporting
This notebook automates the process of analyzing transaction data to generate financial insights.

## 2. Data Understanding
In this phase, we load the 'Online Retail' dataset and inspect its structure, identifying key columns like `InvoiceDate`, `Quantity`, and `UnitPrice`.

In [4]:
import pandas as pd
import numpy as np
import requests
import io

# Using a sample of the Online Retail dataset (Excel/CSV format)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'

print("Downloading dataset... (this may take a minute)")
# Note: Online Retail is an Excel file, we'll use pandas to read it.
# Since downloading directly might be slow, we'll assume the environment has access or use a similar CSV structure for demonstration.
df = pd.read_excel(url)
print("Data loaded successfully!")
display(df.head())

Data loaded successfully!


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Data Cleaning
We need to ensure the `InvoiceDate` is a datetime object and calculate total price per line item.

## 3. Data Preparation
This phase involves cleaning the data: handling missing `CustomerID` values, removing returns (negative quantities), and calculating the `TotalLinePrice` needed for revenue analysis.

In [5]:
# Remove rows without CustomerID and handle negative quantities (returns)
df_clean = df.dropna(subset=['CustomerID'])
df_clean = df_clean[df_clean['Quantity'] > 0]

# Calculate Total Revenue per line
df_clean['TotalLinePrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Extract Year-Month for reporting
df_clean['InvoiceMonth'] = df_clean['InvoiceDate'].dt.to_period('M')

print("Data cleaned.")
display(df_clean.info())

Data cleaned.
<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, 0 to 541908
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   InvoiceNo       397924 non-null  object        
 1   StockCode       397924 non-null  object        
 2   Description     397924 non-null  object        
 3   Quantity        397924 non-null  int64         
 4   InvoiceDate     397924 non-null  datetime64[ns]
 5   UnitPrice       397924 non-null  float64       
 6   CustomerID      397924 non-null  float64       
 7   Country         397924 non-null  object        
 8   TotalLinePrice  397924 non-null  float64       
 9   InvoiceMonth    397924 non-null  period[M]     
dtypes: datetime64[ns](1), float64(3), int64(1), object(4), period[M](1)
memory usage: 33.4+ MB


None

### Automated Monthly Summary
Now we automate the aggregation of revenue by month.

## 4. Modeling & Analysis
In a finance automation context, our 'Model' is the logic used to aggregate data and extract monthly trends.

In [6]:
monthly_revenue = df_clean.groupby('InvoiceMonth')['TotalLinePrice'].sum().reset_index()
monthly_revenue.columns = ['Month', 'MonthlyRevenue']

print("Monthly Revenue Report:")
display(monthly_revenue)

# Export to CSV for the Finance Team
monthly_revenue.to_csv('monthly_finance_report.csv', index=False)
print("Report saved as 'monthly_finance_report.csv'")

Monthly Revenue Report:


,Month,MonthlyRevenue
0,2010-12,572713.890
1,2011-01,569445.040
2,2011-02,447137.350
3,2011-03,595500.760
4,2011-04,469200.361
5,2011-05,678594.560
6,2011-06,661213.690
7,2011-07,600091.011
8,2011-08,645343.900
9,2011-09,952838.382


Report saved as 'monthly_finance_report.csv'


## 5. Evaluation & Deployment
**Evaluation:** We verify that the monthly revenue figures are logical and consistent.
**Deployment:** The automation is 'deployed' by exporting the final result to `monthly_finance_report.csv`, which can be consumed by downstream financial dashboards or stakeholders.

## Automation: Weekly Scheduling
To run this project automatically every week, we can wrap the logic into a single function and use the `schedule` library.

*Note: In a real-world production environment, you would run this script on a cloud server (like GCP or AWS) rather than an interactive notebook.*

In [7]:
!pip install schedule

In [8]:
import schedule
import time

def run_finance_report():
    print(f"Automation started at {time.ctime()}...")
    # 1. Load Data
    df_internal = pd.read_excel(url)

    # 2. Clean Data
    df_clean_internal = df_internal.dropna(subset=['CustomerID'])
    df_clean_internal = df_clean_internal[df_clean_internal['Quantity'] > 0]
    df_clean_internal['TotalLinePrice'] = df_clean_internal['Quantity'] * df_clean_internal['UnitPrice']
    df_clean_internal['InvoiceMonth'] = df_clean_internal['InvoiceDate'].dt.to_period('M')

    # 3. Aggregate and Export
    weekly_report = df_clean_internal.groupby('InvoiceMonth')['TotalLinePrice'].sum().reset_index()
    file_name = f"report_{pd.Timestamp.now().strftime('%Y_%m_%d')}.csv"
    weekly_report.to_csv(file_name, index=False)
    print(f"Report generated: {file_name}")

# Schedule the job for every Monday at 09:00
schedule.every().monday.at("09:00").do(run_finance_report)

print("Scheduler is active. (In a production script, this would run in a continuous loop)")
# To test it immediately, we can call the function once:
# run_finance_report()

Scheduler is active. (In a production script, this would run in a continuous loop)


### Verification
Run the cell below to execute the reporting function manually and verify the output.

In [9]:
# Manually triggering the automated function to verify it works
run_finance_report()

Automation started at Sat Jul 18 09:57:58 2026...
Report generated: report_2026_07_18.csv
